# Prep Data for LME

Merges model outputs, ERP data, and lexical controls into analysis-ready CSVs,
with three corrections applied before merging:

1. **`lang_mae_z`** — language network MAE z-scored within each model.
2. **`cv_layer_surprisal`** — best-layer surprisal via leave-one-dataset-out CV.
3. **`su_sgm_z`** — Lopopolo (2024) Sentence Gestalt Model update, z-scored globally.

**Output:** `lme_data/{model_prefix}_lme.csv` — one file per model, one row per
subject × stimulus. Each file also contains all per-layer surprisal columns
(`layer_00`, `layer_01`, …) for models with tuned-lens output; Qwen3 models have
no layer columns.

**Prerequisites:** Run `extract_controls.ipynb` first to generate `controls.csv`.

In [14]:
import re
import os
import numpy as np
import pandas as pd

In [15]:
ERP_PATH      = 'combined_clean_n400.csv'
CONTROLS_PATH = 'controls.csv'
SGM_PATH      = 'outputs/lopopolo_su_sgm.csv'
OUTPUTS_DIR   = 'outputs/'
LME_DIR       = 'lme_data/'

os.makedirs(LME_DIR, exist_ok=True)

## 1. Load base data

In [16]:
erp      = pd.read_csv(ERP_PATH)
controls = pd.read_csv(CONTROLS_PATH)

# Item-level N400 mean (averaged across subjects) — used only for CV layer selection.
item_n400_mean = erp.groupby('stim_id')['meanAmp_z'].mean()

print(f"ERP:      {erp.shape}  ({erp['subject'].nunique()} subjects, {erp['stim_id'].nunique()} stimuli)")
print(f"Controls: {controls.shape}")
print(f"\nDatasets: {erp['dataset'].value_counts().to_dict()}")

ERP:      (45879, 9)  (98 subjects, 2808 stimuli)
Controls: (2808, 8)

Datasets: {'frank_2015': 37112, 'michaelov_2024': 5526, 'ryskin_2021': 3241}


### Lopopolo SGM metric

`su_sgm` is the Sentence Gestalt Model update from Lopopolo (2024): the mean
absolute error of the gestalt representation before vs. after the critical word.
It is a single fixed metric (not per HuggingFace model) so it is z-scored once
globally across all non-OOV stimuli and merged into every model's output.

OOV items (231 contractions absent from the SGM vocabulary) are set to NaN
before z-scoring so fallback values don't distort the distribution.

In [17]:
sgm = pd.read_csv(SGM_PATH)
sgm.loc[sgm['is_oov'], 'su_sgm'] = np.nan  # OOV fallbacks are unreliable

mu, sigma = sgm['su_sgm'].mean(), sgm['su_sgm'].std(ddof=1)
sgm['su_sgm_z'] = (sgm['su_sgm'] - mu) / sigma

print(f"SGM: {len(sgm)} stimuli, {sgm['is_oov'].sum()} OOV → NaN")
print(f"su_sgm   mean={sgm['su_sgm'].mean():.4f}  std={sgm['su_sgm'].std():.4f}")
print(f"su_sgm_z mean={sgm['su_sgm_z'].mean():.4f}  std={sgm['su_sgm_z'].std():.4f}")

SGM: 2808 stimuli, 231 OOV → NaN
su_sgm   mean=0.0362  std=0.0285
su_sgm_z mean=-0.0000  std=1.0000


## 2. Helper functions

### 2a. Normalise column names
GPT-2 / Pythia metrics CSVs have `n_tokens_x` and `n_tokens_y` (artifact of the
two-pass pipeline merge); Qwen3 CSVs already use `n_tokens`.  Both columns carry
the same value, so we keep one.

In [18]:
def normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise column names across model families."""
    df = df.copy()
    if 'n_tokens_x' in df.columns:
        df = df.rename(columns={'n_tokens_x': 'n_tokens'}).drop(columns=['n_tokens_y'], errors='ignore')
    return df

### 2b. Z-score `lang_mae_update` within model

Language network MAE absolute scale varies with model size, so we z-score
it across all stimuli within each model before comparing across models.

In [19]:
def add_lang_mae_z(df: pd.DataFrame) -> pd.DataFrame:
    """Add lang_mae_z: lang_mae_update z-scored across all stimuli in this model."""
    df = df.copy()
    mu, sigma = df['lang_mae_update'].mean(), df['lang_mae_update'].std(ddof=1)
    df['lang_mae_z'] = (df['lang_mae_update'] - mu) / sigma
    return df

### 2c. CV layer surprisal — leave-one-dataset-out

**Problem:** `pipeline.ipynb` selects the best layer by correlating layer surprisals
with Frank 2015 N400, then evaluates on the same Frank 2015 data — circular.

**Fix:** Leave-one-dataset-out (LODO). For each held-out dataset, the best layer
is chosen using item-level correlation on the *other two* datasets.

Selection criterion: `|Pearson r|` between layer surprisal and mean item-level
`meanAmp_z` (averaged across subjects).  Absolute value is used because the
expected direction is consistently negative, and `|r|` is more stable when
training folds are small.

In [20]:
def add_cv_layer_surprisal(
    df: pd.DataFrame,
    item_n400: pd.Series,
    layer_cols: list[str],
    model_name: str,
) -> pd.DataFrame:
    """
    Leave-one-dataset-out CV layer selection.

    Returns df with two new columns:
        cv_layer_surprisal  — surprisal at the CV-selected layer
        cv_best_layer       — 0-based index of that layer
    """
    df = df.copy()
    df['_n400_mean'] = df['stim_id'].map(item_n400)

    cv_surp  = pd.Series(np.nan, index=df.index)
    cv_layer = pd.array([pd.NA] * len(df), dtype='Int64')

    print(f"  {model_name}  ({len(layer_cols)} layers)")
    for held_out in df['dataset'].unique():
        train = df[df['dataset'] != held_out].dropna(subset=['_n400_mean'])
        test_mask = df['dataset'] == held_out

        corrs = {col: abs(train[col].corr(train['_n400_mean'])) for col in layer_cols}
        best_col = max(corrs, key=corrs.get)
        best_idx = layer_cols.index(best_col)

        cv_surp[test_mask]  = df.loc[test_mask, best_col].values
        cv_layer[test_mask] = best_idx

        r_last = abs(train['last_layer_surprisal'].corr(train['_n400_mean']))
        print(f"    held-out={held_out:20s}  best_layer={best_idx:2d}  |r|={corrs[best_col]:.3f}  (last-layer |r|={r_last:.3f})")

    df['cv_layer_surprisal'] = cv_surp
    df['cv_best_layer']      = cv_layer
    return df.drop(columns=['_n400_mean'])

## 3. Process all models

In [21]:
METRIC_COLS_KEEP = [
    'stim_id', 'dataset', 'critical_word',
    'last_layer_surprisal', 'shallow_surprisal', 'n_tokens',
    'lang_cosine_dist', 'lang_mae_update', 'lang_mae_z',
    'cv_layer_surprisal', 'cv_best_layer',
]

SGM_COLS = ['stim_id', 'su_sgm', 'su_sgm_z']

metrics_files = sorted(
    f for f in os.listdir(OUTPUTS_DIR) if f.endswith('_metrics.csv')
)
print(f"Found {len(metrics_files)} model metrics CSVs\n")

summary_rows = []

for fname in metrics_files:
    model_prefix = fname.replace('_metrics.csv', '')
    metrics = pd.read_csv(os.path.join(OUTPUTS_DIR, fname))
    metrics = normalise_columns(metrics)

    layer_cols = sorted(
        [c for c in metrics.columns if re.match(r'layer_\d+', c)],
        key=lambda c: int(c.split('_')[1])
    )

    # --- Correction 1: z-score lang_mae_update ---
    metrics = add_lang_mae_z(metrics)

    # --- Correction 2: CV layer selection ---
    if layer_cols:
        metrics = add_cv_layer_surprisal(metrics, item_n400_mean, layer_cols, model_prefix)
    else:
        metrics['cv_layer_surprisal'] = np.nan
        metrics['cv_best_layer']      = pd.array([pd.NA] * len(metrics), dtype='Int64')
        print(f"  {model_prefix}  (no tuned-lens layers — cv_layer_surprisal=NaN)")

    # --- Keep named metric columns + all per-layer surprisal columns ---
    keep = [c for c in METRIC_COLS_KEEP if c in metrics.columns] + layer_cols
    metrics_slim = metrics[keep]

    # --- Merge: ERP + metrics + controls + SGM ---
    merged = (
        erp
        .merge(metrics_slim, on=['stim_id', 'dataset'], suffixes=('', '_metrics'))
        .merge(controls.drop(columns=['dataset', 'critical_word']), on='stim_id')
        .merge(sgm[SGM_COLS], on='stim_id', how='left')
    )
    if 'critical_word_metrics' in merged.columns:
        merged = merged.drop(columns=['critical_word_metrics'])

    # Full version (OOV rows have su_sgm_z = NaN)
    out_path = os.path.join(LME_DIR, f'{model_prefix}_lme.csv')
    merged.to_csv(out_path, index=False)

    # OOV-dropped version for su_sgm_z analyses
    merged_no_oov = merged[merged['su_sgm_z'].notna()]
    out_path_no_oov = os.path.join(LME_DIR, f'{model_prefix}_lme_no_oov.csv')
    merged_no_oov.to_csv(out_path_no_oov, index=False)

    summary_rows.append({
        'model':        model_prefix,
        'n_layers':     len(layer_cols),
        'rows':         len(merged),
        'rows_no_oov':  len(merged_no_oov),
        'subjects':     merged['subject'].nunique(),
        'stimuli':      merged['stim_id'].nunique(),
        'has_cv_layer': not metrics['cv_layer_surprisal'].isna().all(),
    })

summary = pd.DataFrame(summary_rows)
print("\n" + summary.to_string(index=False))

Found 18 model metrics CSVs

  eleutherai_pythia_1.4b_deduped  (24 layers)
    held-out=frank_2015            best_layer=20  |r|=0.287  (last-layer |r|=0.289)
    held-out=michaelov_2024        best_layer=22  |r|=0.243  (last-layer |r|=0.229)
    held-out=ryskin_2021           best_layer=18  |r|=0.300  (last-layer |r|=0.288)
  eleutherai_pythia_12b_deduped  (36 layers)
    held-out=frank_2015            best_layer=33  |r|=0.312  (last-layer |r|=0.308)
    held-out=michaelov_2024        best_layer=32  |r|=0.265  (last-layer |r|=0.245)
    held-out=ryskin_2021           best_layer=34  |r|=0.305  (last-layer |r|=0.293)
  eleutherai_pythia_2.8b_deduped  (32 layers)
    held-out=frank_2015            best_layer=27  |r|=0.316  (last-layer |r|=0.300)
    held-out=michaelov_2024        best_layer=30  |r|=0.266  (last-layer |r|=0.241)
    held-out=ryskin_2021           best_layer=30  |r|=0.304  (last-layer |r|=0.290)
  eleutherai_pythia_410m_deduped  (24 layers)
    held-out=frank_2015         

## 4. Inspect one output

In [22]:
example = pd.read_csv(os.path.join(LME_DIR, 'gpt2_lme.csv'))
print("Columns:", list(example.columns))
print(f"\nShape: {example.shape}")
example.head(3)

Columns: ['dataset', 'subject', 'stim_id', 'stim', 'stim_lower', 'condition', 'critical_word', 'meanAmp', 'meanAmp_z', 'last_layer_surprisal', 'shallow_surprisal', 'n_tokens', 'lang_cosine_dist', 'lang_mae_update', 'lang_mae_z', 'cv_layer_surprisal', 'cv_best_layer', 'layer_00', 'layer_01', 'layer_02', 'layer_03', 'layer_04', 'layer_05', 'layer_06', 'layer_07', 'layer_08', 'layer_09', 'layer_10', 'layer_11', 'word_length', 'word_position', 'zipf_freq', 'log10_word_freq', 'coltheart_n', 'su_sgm', 'su_sgm_z']

Shape: (45879, 36)


,dataset,subject,stim_id,stim,stim_lower,condition,critical_word,meanAmp,meanAmp_z,last_layer_surprisal,...,layer_09,layer_10,layer_11,word_length,word_position,zipf_freq,log10_word_freq,coltheart_n,su_sgm,su_sgm_z
0,ryskin_2021,ryskin_s1,0,"He loves me? she wondered, plucking off a dais...","he loves me? she wondered, plucking off a dais...",Synt,petals,2.259037,0.073806,17.665970,...,20.674422,21.788059,31.015973,6,9,3.46,-5.540457,1,0.007342,-1.012549
1,ryskin_2021,ryskin_s1,1,Its alive! he cackled as he animated this mons...,its alive! he cackled as he animated this mons...,Sem,bounty,6.090838,0.530813,18.946643,...,17.877624,17.872257,18.619684,6,9,3.74,-5.259558,1,0.021100,-0.529956
2,ryskin_2021,ryskin_s1,2,"Never stay out past midnight, was his mothers ...","never stay out past midnight, was his mothers ...",SemCrit,admission,1.328902,-0.037128,20.603489,...,15.573940,17.190181,18.385293,9,9,4.15,-4.850750,0,0.014010,-0.778667


In [23]:
example.describe()

,stim_id,meanAmp,meanAmp_z,last_layer_surprisal,shallow_surprisal,n_tokens,lang_cosine_dist,lang_mae_update,lang_mae_z,cv_layer_surprisal,...,layer_09,layer_10,layer_11,word_length,word_position,zipf_freq,log10_word_freq,coltheart_n,su_sgm,su_sgm_z
count,45879.000000,45879.000000,4.587900e+04,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,...,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,45879.000000,44374.000000,44374.000000
mean,1732.645895,0.109296,-1.424833e-17,7.836617,2.773239,1.056061,0.030137,32.796809,0.084711,8.304434,...,8.304434,8.643786,12.499335,4.567536,5.970117,5.533457,-3.466443,14.195710,0.041213,0.175504
std,686.355083,4.790966,9.999782e-01,5.738657,1.898481,0.290506,0.040757,99.544951,1.137168,5.467659,...,5.467659,5.168062,6.661436,2.171293,3.816655,1.336490,1.336236,13.022945,0.029434,1.032413
min,0.000000,-31.762375,-5.618617e+00,0.000464,0.000319,1.000000,0.002240,1.044505,-0.278017,0.002995,...,0.002995,0.000884,1.578089,1.000000,0.000000,0.000000,-9.000000,0.000000,0.003393,-1.151060
25%,1267.000000,-2.693770,-6.343541e-01,3.384134,1.163227,1.000000,0.011142,1.957373,-0.267589,3.953022,...,3.953022,4.660469,8.380587,3.000000,3.000000,4.590000,-4.410039,3.000000,0.016497,-0.691441
50%,1789.000000,-0.097059,-2.333891e-02,6.683736,2.333298,1.000000,0.016803,2.324380,-0.263396,7.411917,...,7.411917,7.951406,11.977411,4.000000,5.000000,5.490000,-3.510040,14.000000,0.033308,-0.101752
75%,2300.000000,2.700145,6.216383e-01,11.011821,4.225444,1.000000,0.026177,2.777488,-0.258220,11.702701,...,11.702701,11.737438,15.483129,6.000000,9.000000,6.510000,-2.489455,22.000000,0.059191,0.806113
max,2807.000000,46.002500,6.746484e+00,43.371020,8.488288,4.000000,0.243600,364.377533,3.872579,46.360642,...,46.360642,41.694470,57.154799,15.000000,21.000000,7.730000,-1.270026,57.000000,0.140698,3.665055


## 5. Sanity checks

### 5a. Confirm z-scored metrics are properly standardised

In [24]:
stim_level = example.drop_duplicates('stim_id')

print(f"lang_mae_z  mean={stim_level['lang_mae_z'].mean():.4f}  std={stim_level['lang_mae_z'].std():.4f}")
print(f"lang_mae_update  mean={stim_level['lang_mae_update'].mean():.2f}  std={stim_level['lang_mae_update'].std():.2f}")
print()
n_oov = stim_level['su_sgm_z'].isna().sum()
print(f"su_sgm_z  mean={stim_level['su_sgm_z'].mean():.4f}  std={stim_level['su_sgm_z'].std():.4f}  ({n_oov} OOV → NaN)")

lang_mae_z  mean=0.0000  std=1.0000
lang_mae_update  mean=25.38  std=87.54

su_sgm_z  mean=-0.0000  std=1.0000  (231 OOV → NaN)


### 5b. Compare CV-selected layer vs. last-layer correlation with N400

In [25]:
stim_level = example.drop_duplicates('stim_id').copy()
stim_level['_n400_mean'] = stim_level['stim_id'].map(item_n400_mean)

rows = []
for ds, g in stim_level.groupby('dataset'):
    r_last = g['last_layer_surprisal'].corr(g['_n400_mean'])
    r_cv   = g['cv_layer_surprisal'].corr(g['_n400_mean']) if g['cv_layer_surprisal'].notna().any() else np.nan
    rows.append({'dataset': ds, 'r_last_layer': r_last, 'r_cv_layer': r_cv,
                 'cv_layer': g['cv_best_layer'].mode()[0] if g['cv_best_layer'].notna().any() else None})

comp = pd.DataFrame(rows)
comp['delta_r'] = comp['r_cv_layer'] - comp['r_last_layer']
print(comp.to_string(index=False))
print("\nNote: correlations are negative (higher surprisal → more negative N400)")

       dataset  r_last_layer  r_cv_layer  cv_layer  delta_r
    frank_2015     -0.261321   -0.246927         9 0.014395
michaelov_2024     -0.313369   -0.301134         9 0.012235
   ryskin_2021     -0.277603   -0.218196         9 0.059407

Note: correlations are negative (higher surprisal → more negative N400)


### 5c. Confirm no data loss in merge

In [26]:
print(f"ERP rows:     {len(erp)}")
print(f"Merged rows:  {len(example)}")
print(f"Rows match:   {len(erp) == len(example)}")
print(f"Missing values per column:")
print(example.isna().sum()[example.isna().sum() > 0].to_string())

ERP rows:     45879
Merged rows:  45879
Rows match:   True
Missing values per column:
su_sgm      1505
su_sgm_z    1505


## 6. LME formula reference

Each `lme_data/{model}_lme.csv` has all columns needed for:

```python
import statsmodels.formula.api as smf

df = pd.read_csv('lme_data/gpt2_lme.csv')

# Full model
full = smf.mixedlm(
    'meanAmp_z ~ last_layer_surprisal + zipf_freq + word_length + n_tokens + word_position + C(dataset)',
    data=df, groups=df['subject'], re_formula='~last_layer_surprisal',
).fit()

# Null model (for LRT)
null = smf.mixedlm(
    'meanAmp_z ~ zipf_freq + word_length + n_tokens + word_position + C(dataset)',
    data=df, groups=df['subject'], re_formula='~last_layer_surprisal',
).fit()

from scipy.stats import chi2
p_val = chi2.sf(2 * (full.llf - null.llf), df=1)
```

Swap `last_layer_surprisal` for `cv_layer_surprisal`, `shallow_surprisal`,
`lang_mae_z`, `lang_cosine_dist`, or `su_sgm_z` to test other predictors.